In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("aqeHandlingSkewApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.skewJoin.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/23 22:30:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Create Products DataFrame with 2 million unique product ids
productsDF = spark.range(1, 2000001).select(
    col("id").alias("ProductID"),
    expr("ROUND(RAND() * 100, 2) AS Price")
)
productsDF.show()

+---------+-----+
|ProductID|Price|
+---------+-----+
|        1| 13.8|
|        2|27.92|
|        3|20.99|
|        4|18.06|
|        5|16.11|
|        6|62.74|
|        7| 9.43|
|        8|78.63|
|        9| 25.7|
|       10|35.06|
|       11| 44.9|
|       12|82.43|
|       13|30.31|
|       14|12.87|
|       15|60.37|
|       16|18.39|
|       17|72.48|
|       18|39.98|
|       19|26.47|
|       20|30.94|
+---------+-----+
only showing top 20 rows



25/06/23 22:30:27 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
# Create Sales DataFrame with 1 million sales records wherein 70% of the records have a ProductID of 1 and a random quantity sold and sale date
salesDF = spark.range(1, 1000001).select(
    col("id").alias("SaleID"),
    expr("""
            CASE
                WHEN RAND() < 0.7
                    THEN 1
                ELSE
                    CAST(RAND() * 2000000 AS INT)
            END
         """).alias("ProductID"),
    expr("CAST(RAND() * 10 AS INTEGER)").alias("QuantitySold"),
    expr("DATE_ADD(CURRENT_DATE(), -CAST(RAND() * 365 AS INT)) AS SaleDate")
)
salesDF.show()

+------+---------+------------+----------+
|SaleID|ProductID|QuantitySold|  SaleDate|
+------+---------+------------+----------+
|     1|        1|           4|2025-04-06|
|     2|   613183|           4|2025-05-01|
|     3|        1|           5|2024-09-20|
|     4|        1|           5|2024-12-06|
|     5|        1|           2|2025-05-08|
|     6|  1511924|           3|2024-08-02|
|     7|        1|           7|2024-06-24|
|     8|   136376|           7|2025-01-15|
|     9|  1358943|           5|2025-03-31|
|    10|        1|           6|2024-07-30|
|    11|        1|           2|2024-12-24|
|    12|        1|           4|2025-02-16|
|    13|        1|           3|2024-12-27|
|    14|  1185449|           6|2024-09-11|
|    15|        1|           3|2025-05-21|
|    16|        1|           1|2024-06-28|
|    17|        1|           4|2024-10-30|
|    18|        1|           6|2025-03-21|
|    19|        1|           2|2024-07-15|
|    20|  1725611|           8|2025-01-02|
+------+---

In [6]:
productsDF.createOrReplaceTempView("Products")
salesDF.createOrReplaceTempView("Sales")

In [7]:
# Check sales of each product
spark.sql("""
    SELECT
        ProductID,
        COUNT(*) AS ProductCount
    FROM
        Sales
    GROUP BY
        ProductID
    ORDER BY
        ProductCount DESC
""").show()

+---------+------------+
|ProductID|ProductCount|
+---------+------------+
|        1|      700073|
|  1881770|           5|
|  1237318|           4|
|  1477941|           4|
|     5136|           4|
|  1741980|           4|
|  1759651|           4|
|   398788|           4|
|  1091330|           4|
|  1989222|           4|
|  1515505|           4|
|  1876240|           4|
|   512992|           4|
|  1279773|           4|
|  1729693|           4|
|   771016|           4|
|    17010|           4|
|   552465|           4|
|  1514716|           4|
|  1700874|           4|
+---------+------------+
only showing top 20 rows



In [8]:
# Find total number of products sold per day
spark.sql("""
    SELECT
        s.SaleDate,
        SUM(Price * QuantitySold) AS SalesAmount
    FROM
        Sales s
    JOIN
        Products p ON s.ProductID = p.ProductID
    GROUP BY
        s.SaleDate
    ORDER BY
        SalesAmount DESC
""").show()

+----------+------------------+
|  SaleDate|       SalesAmount|
+----------+------------------+
|2025-01-22| 331909.3699999997|
|2024-12-19|326738.40999999986|
|2024-07-26|326516.06000000023|
|2025-03-10|326212.49999999994|
|2024-10-27|325919.53999999975|
|2024-12-06| 325642.5300000001|
|2024-06-29| 324158.9399999999|
|2025-04-10| 324073.0500000003|
|2025-02-23|         323702.84|
|2024-09-16| 323271.8999999998|
|2024-07-03|322263.51999999996|
|2025-01-14|321307.78999999975|
|2024-12-27|320649.07000000024|
|2025-06-13| 320351.1299999999|
|2025-02-17|320304.29000000015|
|2025-02-18|320087.92999999993|
|2025-05-26| 319951.3699999999|
|2024-10-07| 319521.4099999999|
|2025-04-08| 319295.0499999998|
|2024-09-03|319037.58999999997|
+----------+------------------+
only showing top 20 rows



In [9]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [ ]:
spark.sql("""
    SELECT
        s.SaleDate,
        SUM(Price * QuantitySold) AS SalesAmount
    FROM
        Sales s
    JOIN
        Products p ON s.ProductID = p.ProductID
    GROUP BY
        s.SaleDate
    ORDER BY
        SalesAmount DESC
""").show()

+----------+------------------+
|  SaleDate|       SalesAmount|
+----------+------------------+
|2025-01-22| 331909.3699999997|
|2024-12-19|326738.40999999986|
|2024-07-26|326516.06000000023|
|2025-03-10|326212.49999999994|
|2024-10-27|325919.53999999975|
|2024-12-06| 325642.5300000001|
|2024-06-29| 324158.9399999999|
|2025-04-10| 324073.0500000003|
|2025-02-23|         323702.84|
|2024-09-16| 323271.8999999998|
|2024-07-03|322263.51999999996|
|2025-01-14|321307.78999999975|
|2024-12-27|320649.07000000024|
|2025-06-13| 320351.1299999999|
|2025-02-17|320304.29000000015|
|2025-02-18|320087.92999999993|
|2025-05-26| 319951.3699999999|
|2024-10-07| 319521.4099999999|
|2025-04-08| 319295.0499999998|
|2024-09-03|319037.58999999997|
+----------+------------------+
only showing top 20 rows



25/06/24 06:06:05 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1053720 ms exceeds timeout 120000 ms
25/06/24 06:06:05 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/24 06:06:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$